In [2]:
!pip install gensim

In [3]:
import pandas as pd
import numpy as np
import re
import string
import unicodedata
from collections import Counter
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.tokenize import word_tokenize, TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.chunk import ne_chunk
from nltk.tag import pos_tag
from nltk.tree import Tree

# Emoji handling
import emoji

# Spell correction and contractions
from textblob import TextBlob
import contractions

# Multiprocessing for speed
from multiprocessing import Pool, cpu_count
from functools import lru_cache
from tqdm import tqdm

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Deep learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Word embeddings
from gensim.models import Word2Vec
import fasttext

ModuleNotFoundError: No module named 'gensim'

In [ ]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Load Data
corpus_df = pd.read_csv("./Sentiment_Data.csv", encoding="ISO-8859-1")

In [ ]:
# Show data overview
print("Dataset Overview:")
print(f"Dataset shape: {corpus_df.shape}")
print(f"Columns: {corpus_df.columns.tolist()}")
print("\nSentiment Distribution:")
print(corpus_df['Sentiment'].value_counts())

In [ ]:
# Merge sentiment categories
def merge_sentiments(sentiment):
    if sentiment in ['Mild_Pos', 'Strong_Pos']:
        return 'Positive'
    elif sentiment in ['Mild_Neg', 'Strong_Neg']:
        return 'Negative'
    else:
        return 'Neutral'

corpus_df['Sentiment_Merged'] = corpus_df['Sentiment'].apply(merge_sentiments)
print("\nMerged Sentiment Distribution:")
print(corpus_df['Sentiment_Merged'].value_counts())

In [ ]:
# Balance dataset
sample_size = 25000
print(f"Balancing dataset with {sample_size} samples per class.")

def balance_dataset(df, target_col='Sentiment_Merged', min_count=None):
    if min_count is None:
        min_count = df[target_col].value_counts().min()
    
    balanced_dfs = []
    for sentiment in df[target_col].unique():
        sentiment_df = df[df[target_col] == sentiment]
        if len(sentiment_df) > min_count:
            balanced_df = sentiment_df.sample(n=min_count, random_state=42)
        else:
            balanced_df = sentiment_df
        balanced_dfs.append(balanced_df)
    
    return pd.concat(balanced_dfs, ignore_index=True)

balanced_corpus = balance_dataset(corpus_df, min_count=sample_size)
print(f"Balanced dataset shape: {balanced_corpus.shape}")

In [ ]:
balanced_corpus = balance_dataset(corpus_df, min_count=sample_size)
print(f"Balanced dataset shape: {balanced_corpus.shape}")
print("Balanced sentiment distribution:")
print(balanced_corpus['Sentiment_Merged'].value_counts())

In [ ]:
# Initialize preprocessing tools
tweet_tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [ ]:
# Create abbreviation dictionar
abbreviations = {
    "u": "you",
    "ur": "your",
    "n": "and",
    "2": "to",
    "4": "for",
    "w/": "with",
    "w/o": "without",
    "thru": "through",
    "tho": "though",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of",
    "outta": "out of",
    "dunno": "don't know",
    "gimme": "give me",
    "lemme": "let me",
    "btw": "by the way",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "rofl": "rolling on floor laughing",
    "smh": "shaking my head",
    "tbh": "to be honest",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "aka": "also known as",
    "asap": "as soon as possible",
    "fyi": "for your information",
    "diy": "do it yourself",
    "faq": "frequently asked questions",
    "irl": "in real life",
    "ppl": "people",
    "msg": "message",
    "txt": "text",
    "pic": "picture",
    "vid": "video",
    "app": "application",
    "tech": "technology",
    "biz": "business",
    "edu": "education",
    "gov": "government",
    "org": "organization",
    "info": "information",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "btw": "by the way",
    "idk": "i do not know",
    "smh": "shaking my head",
    "afaik": "as far as i know",
    "tbh": "to be honest",
    "imo": "in my opinion",
    "icymi": "in case you missed it",
    "fwiw": "for what it is worth",
    "ftw": "for the win",
    "lmk": "let me know",
    "rn": "right now",
    "thx": "thanks",
    "til": "today i learned",
    "brb": "be right back",
    "gg": "good game",
    "noob": "newbie",
    "ootd": "outfit of the day",
    "fyp": "for you page",
    "hmu": "hit me up",
    "iiuc": "if i understand correctly",
    "ikr": "i know, right",
    "irl": "in real life",
    "iss": "i am so sorry",
    "jsyk": "just so you know",
    "lowkey": "quietly",
    "highkey": "obviously",
    "ngl": "not gonna lie",
    "oot": "out of the",
    "pls": "please",
    "rizz": "charisma",
    "ship": "support a romantic relationship",
    "slay": "do something well",
    "s/o": "shoutout",
    "stan": "support",
    "tbf": "to be fair",
    "tea": "gossip",
    "vibe check": "evaluation of mood",
    "wtf": "what the freak",
    "wym": "what you mean",
    "yaaas": "strong agreement",
    "cc": "carbon-copy",
    "cx": "correction",
    "ct": "cut tweet",
    "dm": "direct message",
    "ht": "hat tip",
    "mt": "modified tweet",
    "prt": "please retweet",
    "rt": "retweet",
    "em": "email marketing",
    "ezine": "electronic magazine",
    "fb": "facebook",
    "li": "linkedin",
    "seo": "search engine optimization",
    "sm": "social media",
    "smm": "social media marketing",
    "smo": "social media optimization",
    "sn": "social network",
    "sroi": "social return on investment",
    "ugc": "user generated content",
    "yt": "youtube",
    "ab/abt": "about",
    "b4": "before",
    "bfn": "bye for now",
    "bgd": "background",
    "bh": "blockhead",
    "br": "best regards",
    "cd9": "code 9",
    "chk": "check",
    "cul8r": "see you later",
    "dam": "don not annoy me",
    "dd": "dear daughter",
    "df": "dear fiancé",
    "dp": "profile pic",
    "ds": "dear son",
    "dyk": "did you know, do you know",
    "ema": "email address",
    "ftf": "face to face",
    "f2f": "face to face",
    "ff": "follow friday",
    "fotd": "find of the day",
    "gts": "guess the song",
    "hagn": "have a good night",
    "hand": "have a nice day",
    "hotd": "headline of the day",
    "hth": "hope that helps",
    "ic": "i see",
    "iirc": "if i remember correctly",
    "jk": "just kidding, joke",
    "jv": "joint venture",
    "kk": "ok got it",
    "kyso": "knock your socks off",
    "lhh": "laugh hella hard",
    "lmao": "laughing my ass off",
    "lo": "little one",
    "mm": "music monday",
    "mirl": "meet in real life",
    "nbd": "no big deal",
    "nct": "nobody cares, though",
    "nfw": "no freaking way",
    "njoy": "enjoy",
    "nsfw": "not safe for work",
    "nts": "note to self",
    "oh": "overheard",
    "omfg": "oh my freaking god",
    "oomf": "one of my followers",
    "orly": "oh really",
    "plmk": "please let me know",
    "pnp": "party and play",
    "qotd": "quote of the day",
    "re": "in reply to, in regards to",
    "rlrt": "real-life re-tweet",
    "rtq": "read the question",
    "sfw": "safe for work",
    "smdh": "shaking my damn head",
    "so": "significant other",
    "srs": "serious",
    "tftf": "thanks for the follow",
    "tf": "thanks for this tweet",
    "tj": "tweetjack",
    "tl": "timeline",
    "tldr": "too long, did not read",
    "tmb": "tweet me back",
    "tt": "trending topic",
    "tyia": "thank you in advance",
    "tyt": "take your time",
    "tyvw": "thank you very much",
    "w/": "with",
    "w/e": "weekend",
    "wtv": "whatever",
    "ygtr": "you got that right",
    "ykwim": "you know what i mean",
    "ykyat": "you know you are addicted to",
    "ymmv": "your mileage may vary",
    "yolo": "you only live once",
    "yoyo": "you are on your own",
    "yw": "you are welcome",
    "zomg": "omg to the max"
}

In [ ]:
# Precompile regex patterns for efficiency
PATTERNS = {
    'urls': re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE),
    'mentions': re.compile(r'@\w+'),
    'hashtags': re.compile(r'#(\w+)'),
    'numbers': re.compile(r'\b\d+\b'),
    'repeated_chars': re.compile(r'(.)\1{2,}'),
    'non_alpha': re.compile(r'[^a-zA-Z\s]'),
    'extra_spaces': re.compile(r'\s+'),
    'short_words': re.compile(r'\b\w{1,2}\b')
}

### Text Cleaning Functions

In [ ]:
def extract_named_entities(text):
    try:
        tokens = word_tokenize(text)
        pos_tags = pos_tag(tokens)
        chunks = ne_chunk(pos_tags)
        
        entities = []
        for chunk in chunks:
            if isinstance(chunk, Tree):
                entity = ' '.join([token for token, pos in chunk.leaves()])
                entities.append((entity, chunk.label()))
        
        return entities
    except:
        return []

def comprehensive_preprocess_text(text):
    
    # Handle null/empty text
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return {
            'original_text': text,
            'processed_text': '',
            'tokens': [],
            'lemmatized_tokens': [],
            'stemmed_tokens': [],
            'named_entities': [],
            'token_count': 0,
            'char_count': 0,
            'emoji_count': 0,
            'mention_count': 0,
            'hashtag_count': 0,
            'url_count': 0
        }
    
    original_text = text
    
    # Step 1: Process emojis - convert to text descriptions
    emoji_count = len(re.findall(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]', text))
    try:
        text = emoji.demojize(text, delimiters=(" ", " "))
    except:
        pass
    
    # Count special elements before removal
    url_count = len(PATTERNS['urls'].findall(text))
    mention_count = len(PATTERNS['mentions'].findall(text))
    hashtag_count = len(PATTERNS['hashtags'].findall(text))
    
    # Step 2: Remove/replace non-grammatical symbols
    text = PATTERNS['urls'].sub(' URL ', text)
    text = PATTERNS['mentions'].sub(' MENTION ', text)
    text = PATTERNS['hashtags'].sub(r' \1 ', text)  # Keep hashtag content
    text = PATTERNS['numbers'].sub(' NUMBER ', text)
    
    # Step 3: Run Named Entity Recognition (before heavy cleaning)
    named_entities = extract_named_entities(text)
    
    # Step 4: Spell correction (limited to avoid slowdown)
    try:
        # Only correct words that are likely misspelled (short correction)
        blob = TextBlob(text)
        text = str(blob.correct())
    except:
        pass
    
    # Step 5: Handle abbreviations and contractions
    text = text.lower()
    try:
        text = contractions.fix(text)
    except:
        pass
    
    # Expand abbreviations
    words = text.split()
    expanded_words = []
    for word in words:
        if word in abbreviations:
            expanded_words.append(abbreviations[word])
        else:
            expanded_words.append(word)
    text = ' '.join(expanded_words)
    
    # Step 6: Remove non-grammatical punctuation
    text = PATTERNS['repeated_chars'].sub(r'\1\1', text)  # Reduce repeated chars
    text = PATTERNS['non_alpha'].sub(' ', text)
    text = PATTERNS['extra_spaces'].sub(' ', text).strip()
    
    # Step 7: Tokenization
    try:
        tokens = tweet_tokenizer.tokenize(text)
        # Filter tokens: keep only alphabetic, length > 2, not stopwords
        tokens = [token for token in tokens if token.isalpha() and len(token) > 2 and token not in stop_words]
    except:
        tokens = [word for word in text.split() if word.isalpha() and len(word) > 2 and word not in stop_words]
    
    # Step 8: Lemmatization and Stemming
    lemmatized_tokens = []
    stemmed_tokens = []
    
    for token in tokens:
        try:
            lemmatized = lemmatizer.lemmatize(token)
            stemmed = stemmer.stem(token)
            lemmatized_tokens.append(lemmatized)
            stemmed_tokens.append(stemmed)
        except:
            lemmatized_tokens.append(token)
            stemmed_tokens.append(token)
    
    # Create final processed text
    processed_text = ' '.join(lemmatized_tokens)
    
    return {
        'original_text': original_text,
        'processed_text': processed_text,
        'tokens': tokens,
        'lemmatized_tokens': lemmatized_tokens,
        'stemmed_tokens': stemmed_tokens,
        'named_entities': named_entities,
        'token_count': len(lemmatized_tokens),
        'char_count': len(processed_text),
        'emoji_count': emoji_count,
        'mention_count': mention_count,
        'hashtag_count': hashtag_count,
        'url_count': url_count
    }

### Model-Specific Preprocessing Functions

In [ ]:
def process_chunk(chunk):
    """Process a chunk of texts"""
    results = []
    for text in chunk:
        try:
            result = comprehensive_preprocess_text(text)
            results.append(result)
        except Exception as e:
            # Return empty result for failed processing
            results.append({
                'original_text': text,
                'processed_text': '',
                'tokens': [],
                'lemmatized_tokens': [],
                'stemmed_tokens': [],
                'named_entities': [],
                'token_count': 0,
                'char_count': 0,
                'emoji_count': 0,
                'mention_count': 0,
                'hashtag_count': 0,
                'url_count': 0
            })
    return results

def sequential_preprocess_dataset(texts):
    """Process the dataset sequentially (fallback method)"""
    print("Using sequential processing...")
    results = []
    for text in tqdm(texts, desc="Processing tweets sequentially"):
        try:
            result = comprehensive_preprocess_text(text)
            results.append(result)
        except Exception as e:
            print(f"Error processing text: {e}")
            # Return empty result for failed processing
            results.append({
                'original_text': text,
                'processed_text': '',
                'tokens': [],
                'lemmatized_tokens': [],
                'stemmed_tokens': [],
                'named_entities': [],
                'token_count': 0,
                'char_count': 0,
                'emoji_count': 0,
                'mention_count': 0,
                'hashtag_count': 0,
                'url_count': 0
            })
    return results

def parallel_preprocess_dataset(df, n_workers=None, force_sequential=False):
    """Process the dataset with multiprocessing or sequential processing"""
    
    # Reset index and get texts
    df = df.reset_index(drop=True)
    texts = df['Tweet'].tolist()
    
    # Option 1: Force sequential processing
    if force_sequential:
        print("Sequential processing forced by user...")
        return sequential_preprocess_dataset(texts)
    
    # Option 2: Auto-detect small datasets
    if len(texts) < 1000:
        print("Small dataset detected. Using sequential processing...")
        return sequential_preprocess_dataset(texts)
    
    # Option 3: Try parallel processing first
    if n_workers is None:
        n_workers = min(cpu_count(), 4)  # Conservative for stability
    
    print(f"Attempting parallel processing with {n_workers} workers...")
    
    # Use appropriate batch size
    batch_size = max(500, len(texts) // (n_workers * 2))
    chunks = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    
    print(f"Processing {len(chunks)} chunks of ~{batch_size} tweets each...")
    
    try:
        # Test multiprocessing capability first
        print("Testing multiprocessing capability...")
        test_chunk = chunks[0][:10]  # Test with first 10 items
        
        with Pool(n_workers) as pool:
            test_result = pool.apply(process_chunk, (test_chunk,))
            
        if test_result:
            print("✓ Multiprocessing test successful! Starting full parallel processing...")
            
            # Full parallel processing
            with Pool(n_workers) as pool:
                chunk_results = []
                for result in tqdm(pool.imap(process_chunk, chunks), total=len(chunks), desc="Processing chunks in parallel"):
                    chunk_results.extend(result)
            
            print("✓ Parallel processing completed successfully!")
            return chunk_results
        else:
            raise Exception("Multiprocessing test failed")
            
    except Exception as e:
        print(f" Parallel processing failed: {e}")
        print(" Switching to sequential processing...")
        return sequential_preprocess_dataset(texts)

In [ ]:
# Process the dataset with automatic fallback
print(f"\nStarting comprehensive preprocessing of {len(balanced_corpus):,} tweets...")
start_time = pd.Timestamp.now()

# Option to force sequential processing (set to True if you want to skip parallel processing)
FORCE_SEQUENTIAL = False  # Change this to True to force sequential processing

processed_results = parallel_preprocess_dataset(balanced_corpus, n_workers=None, force_sequential=FORCE_SEQUENTIAL)

end_time = pd.Timestamp.now()
processing_time = (end_time - start_time).total_seconds()
print(f"Preprocessing completed in {processing_time:.2f} seconds!")

# Create comprehensive dataframe
print("\nCreating comprehensive dataframe...")
result_df = pd.DataFrame(processed_results)

# Combine with original data
final_df = pd.concat([balanced_corpus.reset_index(drop=True), result_df], axis=1)

# Print statistics
print(f"\nPREPROCESSING STATISTICS:")
print(f"✓ Total tweets processed: {len(final_df):,}")
print(f"✓ Average tokens per tweet: {final_df['token_count'].mean():.1f}")
print(f"✓ Average characters per processed tweet: {final_df['char_count'].mean():.1f}")
print(f"✓ Processing speed: {len(final_df)/processing_time:.0f} tweets/second")
print(f"✓ Average emojis per tweet: {final_df['emoji_count'].mean():.1f}")
print(f"✓ Average mentions per tweet: {final_df['mention_count'].mean():.1f}")
print(f"✓ Average hashtags per tweet: {final_df['hashtag_count'].mean():.1f}")

# Calculate vocabulary size
all_tokens = []
for tokens in final_df['lemmatized_tokens']:
    all_tokens.extend(tokens)
vocab_size = len(set(all_tokens))
print(f"✓ Vocabulary size: {vocab_size:,}")

### Train-Test Split for Each Model

In [ ]:
# Create train-test split
print("\nCreating train-test split...")
X = final_df.drop(['Sentiment', 'Sentiment_Merged'], axis=1)
y = final_df['Sentiment_Merged']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Compute class weights for imbalanced learning
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

print(f"✓ Train set: {len(X_train):,} tweets")
print(f"✓ Test set: {len(X_test):,} tweets")
print(f"✓ Class weights: {class_weight_dict}")

# Create model-specific data formats
print("\nCreating model-specific data formats...")

# Common parameters
vocab_size_limit = 10000
max_sequence_length = 250
embedding_dim = 128

# 1. For LSTM models - use processed_text and lemmatized_tokens
print("   - Preparing LSTM data...")
lstm_tokenizer = Tokenizer(num_words=vocab_size_limit, oov_token="<OOV>")
lstm_tokenizer.fit_on_texts(X_train['processed_text'])

X_train_lstm = lstm_tokenizer.texts_to_sequences(X_train['processed_text'])
X_test_lstm = lstm_tokenizer.texts_to_sequences(X_test['processed_text'])

X_train_lstm = pad_sequences(X_train_lstm, maxlen=max_sequence_length, truncating='post')
X_test_lstm = pad_sequences(X_test_lstm, maxlen=max_sequence_length, truncating='post')

# 2. For Transformer models - use tokens
print("   - Preparing Transformer data...")
transformer_tokenizer = Tokenizer(num_words=vocab_size_limit, oov_token="<OOV>")
all_tokens_text = [' '.join(tokens) for tokens in X_train['lemmatized_tokens']]
transformer_tokenizer.fit_on_texts(all_tokens_text)

X_train_transformer = transformer_tokenizer.texts_to_sequences(all_tokens_text)
X_test_transformer_text = [' '.join(tokens) for tokens in X_test['lemmatized_tokens']]
X_test_transformer = transformer_tokenizer.texts_to_sequences(X_test_transformer_text)

X_train_transformer = pad_sequences(X_train_transformer, maxlen=max_sequence_length, truncating='post')
X_test_transformer = pad_sequences(X_test_transformer, maxlen=max_sequence_length, truncating='post')

# 3. Train Word2Vec model for embeddings
print("   - Training Word2Vec model...")
w2v_sentences = X_train['lemmatized_tokens'].tolist()
w2v_model = Word2Vec(
    sentences=w2v_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=2,
    sg=1,  # Skip-gram
    workers=4,
    epochs=10
)

# Create embedding matrix for Word2Vec
word_index = lstm_tokenizer.word_index
w2v_embedding_matrix = np.zeros((min(len(word_index) + 1, vocab_size_limit), embedding_dim))
for word, i in word_index.items():
    if i < vocab_size_limit:
        try:
            w2v_embedding_matrix[i] = w2v_model.wv[word]
        except KeyError:
            w2v_embedding_matrix[i] = np.random.normal(0, 0.1, embedding_dim)

# 4. Convert labels to categorical for neural networks
y_train_categorical = to_categorical(y_train_encoded, num_classes=len(np.unique(y_train_encoded)))
y_test_categorical = to_categorical(y_test_encoded, num_classes=len(np.unique(y_test_encoded)))

### Save Processed Datasets & Processing Tools

In [ ]:
# Save all preprocessed data
print("\nSaving preprocessed data...")
save_dir = './unified_preprocessed_data/'
os.makedirs(save_dir, exist_ok=True)

# Save main datasets
final_df.to_csv(f"{save_dir}complete_preprocessed_data.csv", index=False)
X_train.to_csv(f"{save_dir}X_train.csv", index=False)
X_test.to_csv(f"{save_dir}X_test.csv", index=False)

# Save labels
np.save(f"{save_dir}y_train_encoded.npy", y_train_encoded)
np.save(f"{save_dir}y_test_encoded.npy", y_test_encoded)
np.save(f"{save_dir}y_train_categorical.npy", y_train_categorical)
np.save(f"{save_dir}y_test_categorical.npy", y_test_categorical)

# Save model-specific data
print("   - Saving model-specific datasets...")

# LSTM data
np.save(f"{save_dir}X_train_lstm.npy", X_train_lstm)
np.save(f"{save_dir}X_test_lstm.npy", X_test_lstm)

# Transformer data
np.save(f"{save_dir}X_train_transformer.npy", X_train_transformer)
np.save(f"{save_dir}X_test_transformer.npy", X_test_transformer)

# Word2Vec embedding matrix
np.save(f"{save_dir}w2v_embedding_matrix.npy", w2v_embedding_matrix)

# Save tokenizers
with open(f"{save_dir}lstm_tokenizer.pkl", 'wb') as f:
    pickle.dump(lstm_tokenizer, f)

with open(f"{save_dir}transformer_tokenizer.pkl", 'wb') as f:
    pickle.dump(transformer_tokenizer, f)

# Save Word2Vec model
w2v_model.save(f"{save_dir}word2vec_model.model")

# Save label encoder
with open(f"{save_dir}label_encoder.pkl", 'wb') as f:
    pickle.dump(label_encoder, f)

# Save configuration
config = {
    'vocab_size_limit': vocab_size_limit,
    'max_sequence_length': max_sequence_length,
    'embedding_dim': embedding_dim,
    'class_weight_dict': class_weight_dict,
    'processing_time': processing_time,
    'vocab_size': vocab_size,
    'num_classes': len(np.unique(y_train_encoded)),
    'class_names': label_encoder.classes_.tolist()
}

with open(f"{save_dir}config.pkl", 'wb') as f:
    pickle.dump(config, f)

In [ ]:
# Create summary
summary = f"""
PREPROCESSING SUMMARY
================================

Dataset Information:
- Total tweets: {len(final_df):,}
- Train set: {len(X_train):,}
- Test set: {len(X_test):,}
- Vocabulary size: {vocab_size:,}
- Processing time: {processing_time:.2f} seconds
- Processing speed: {len(final_df)/processing_time:.0f} tweets/second

Preprocessing Steps Completed:
✓ Emoji processing (converted to text)
✓ Non-grammatical symbol removal
✓ Named Entity Recognition
✓ Spell correction
✓ Abbreviation expansion
✓ Contraction expansion
✓ Punctuation cleaning
✓ Tokenization
✓ Lemmatization and stemming
✓ Stopword removal

Model-Ready Data Created:
✓ LSTM data: X_train_lstm.npy, X_test_lstm.npy
✓ Transformer data: X_train_transformer.npy, X_test_transformer.npy
✓ Word2Vec embeddings: w2v_embedding_matrix.npy
✓ Categorical labels: y_train_categorical.npy, y_test_categorical.npy

Ready for Models:
1. ✓ Bidirectional LSTM with learnable embeddings
2. ✓ Causal transformer with learnable embeddings
3. ✓ Causal transformer with FastText/ELMo embeddings
4. ✓ Non-causal transformer with Word2Vec embeddings
5. ✓ Non-causal transformer with learnable embeddings

Configuration saved in: config.pkl
Class weights: {class_weight_dict}
"""

with open(f"{save_dir}SUMMARY.txt", 'w') as f:
    f.write(summary)

print(f"\n{summary}")
print("PREPROCESSING COMPLETE")
print(f" All data saved to: {save_dir}")